# Agriculture AI: Crop Yield Prediction and Dataset Question Answering

**Course:** Fundamentals of AI Using Agriculture Data Set — ANNAM.AI, IIT Ropar  
**Student:** [Enter your name]  
**College:** [Enter your college]  
**Note:** This starter project uses a **synthetic educational dataset** generated for demonstrating the workflow. It is not real-time government data and must not be used for actual farming decisions.

## Objective
Explore how plot size, rainfall, fertilizer quantity, and average temperature relate to crop yield, train a regression model to estimate yield, and answer basic questions about the dataset.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully.")

## 2. Load the CSV dataset
Upload `Agriculture_AI_Dataset.csv` to this Colab session when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose Agriculture_AI_Dataset.csv

# Read the uploaded CSV file
csv_name = next(name for name in uploaded.keys() if name.lower().endswith(".csv"))
df = pd.read_csv(csv_name)
df.head()

In [ ]:
# 3. Understand the dataset
print("Dataset shape (rows, columns):", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types and non-null counts:")
df.info()
print("\\nMissing values per column:")
print(df.isnull().sum())
print("\\nDescriptive statistics:")
display(df.describe(numeric_only=True))

In [ ]:
# 4. Basic data cleaning
# Remove duplicate records and rows with missing target values.
df = df.drop_duplicates()
df = df.dropna(subset=["Yield_quintal"])
# Fill missing predictor values with each column's median.
predictor_cols = ["Plot_Size_acre", "Rainfall_mm", "Fertilizer_kg", "Avg_Temp_C"]
for col in predictor_cols:
    df[col] = df[col].fillna(df[col].median())

print("Shape after cleaning:", df.shape)
print("Remaining missing values:")
print(df.isnull().sum())

In [ ]:
# 5. Exploratory data analysis
display(df[predictor_cols + ["Yield_quintal"]].corr(numeric_only=True).round(2))

df["Yield_quintal"].hist(bins=20)
plt.title("Distribution of Crop Yield")
plt.xlabel("Yield (quintals)")
plt.ylabel("Number of farmers/plots")
plt.show()

for col in predictor_cols:
    plt.figure()
    plt.scatter(df[col], df["Yield_quintal"], alpha=0.7)
    plt.title(f"Yield vs {col}")
    plt.xlabel(col)
    plt.ylabel("Yield (quintals)")
    plt.show()

In [ ]:
# 6. Prepare training and testing data
features = ["Plot_Size_acre", "Rainfall_mm", "Fertilizer_kg", "Avg_Temp_C"]
target = "Yield_quintal"

X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

In [ ]:
# 7. Train a Random Forest regression model
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
print("Model training completed.")

In [ ]:
# 8. Evaluate the model
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f} quintals")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} quintals")
print(f"R-squared (R²): {r2:.3f}")

results = pd.DataFrame({"Actual_Yield": y_test.values, "Predicted_Yield": predictions.round(2)})
display(results.head(10))

In [ ]:
# 9. Feature importance
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
display(importance.to_frame("Importance"))
importance.sort_values().plot(kind="barh", title="Feature Importance")
plt.xlabel("Importance score")
plt.show()

## 10. Ask questions about the dataset
This small rule-based function answers summary questions from the loaded dataset and uses the trained model for a yield prediction. It is a simple dataset question-answering interface, not a general-purpose language model.

In [ ]:
def answer_question(question):
    q = question.lower().strip()

    if "number of rows" in q or "how many rows" in q or "dataset size" in q:
        return f"The dataset has {len(df)} rows and {len(df.columns)} columns."
    if "average yield" in q or "mean yield" in q:
        return f"The average recorded yield is {df['Yield_quintal'].mean():.2f} quintals."
    if "highest yield" in q or "maximum yield" in q:
        row = df.loc[df["Yield_quintal"].idxmax()]
        return f"The highest yield is {row['Yield_quintal']:.2f} quintals (Farmer ID: {row['Farmer_ID']})."
    if "average rainfall" in q or "mean rainfall" in q:
        return f"The average rainfall is {df['Rainfall_mm'].mean():.1f} mm."
    if "average temperature" in q or "mean temperature" in q:
        return f"The average temperature is {df['Avg_Temp_C'].mean():.1f} °C."
    if "average fertilizer" in q or "mean fertilizer" in q:
        return f"The average fertilizer quantity is {df['Fertilizer_kg'].mean():.1f} kg."

    if "predict yield" in q or "estimate yield" in q:
        print("Enter values for a single plot:")
        plot_size = float(input("Plot size (acre): "))
        rainfall = float(input("Rainfall (mm): "))
        fertilizer = float(input("Fertilizer (kg): "))
        temp = float(input("Average temperature (°C): "))
        sample = pd.DataFrame([[plot_size, rainfall, fertilizer, temp]], columns=features)
        estimate = model.predict(sample)[0]
        return f"Estimated yield: {estimate:.2f} quintals. This is a model estimate, not a farming guarantee."

    return ("I can answer questions about number of rows, average/highest yield, "
            "average rainfall, average temperature, average fertilizer, or predict yield.")

question = input("Ask a question about the dataset: ")
print(answer_question(question))

## 11. Conclusion and limitations
The notebook demonstrates dataset inspection, cleaning, visual analysis, regression model training, evaluation, and simple question answering. Because this dataset is synthetic, model performance only demonstrates the workflow and does not establish real-world agricultural accuracy. For practical use, replace it with a documented real dataset and validate the model with domain experts.

## References
1. scikit-learn documentation: https://scikit-learn.org/stable/
2. pandas documentation: https://pandas.pydata.org/docs/
3. Matplotlib documentation: https://matplotlib.org/stable/
4. Dataset: synthetic educational dataset generated for this project; no external source.